# This notebook compares the results for SATNAC 2025

The original code was fairly deprecated, so the configs were recently re-run, see [configfiles/Satnac_2025](../../configfiles/Satnac_2025/README.md) for more details. 

There are may be runs with similar configs, a simple filter was applied to the Wandb project, as shown in [filters.py](./filters.py).

Also see the [companion notebook](./benchmark.ipynb) which benchmarks the GPU utilisation of the models presented in this notebook. 

In [14]:
# locals
import pandas as pd

# from src.configs import get_avail_splits
from src.results import fetch_runs
from src.run_types import CUTOFF_9_NAMES, RESULTS_DIR


In [ ]:
results_dir = RESULTS_DIR / 'satnac_2025'
# runs = load_runs(results_dir / 'runs.json')
runs = fetch_runs(results_dir / 'filters.py')
print(f'{len(runs)} found')

11 found


## Now we can compare the runs

In [16]:
avail_acc_types = ["top_k_average_per_class_acc", "top_k_per_instance_acc"]
acc_type = avail_acc_types[1]
set_names = ['test', 'val']

#### Convert to DataFrame

In [17]:
df_format = [
    {
        "model": run.admin.model,
        "exp no": run.admin.exp_no,
        "run_id": run.wandb.run_id,
        "No. frames": run.data.target_length,
        "subset": run.admin.split,
    }
    | {f'{set_name} {k}': v for set_name in set_names for k, v in run.results.model_dump()[set_name][acc_type].items()}
    | {
        "best_val_acc": run.results.best_val_acc,
        "best_val_loss": run.results.best_val_loss,
        "test_loss": run.results.test.average_loss,
        "config path": run.admin.config_path,
        "weight path": run.admin.weight_path,
        }
    for run in runs
]


df = pd.DataFrame(df_format)

In [18]:
ns = [1, 5, 10] #topn
for set_name in set_names:
    for n in ns:
        old_name, new_name = f"{set_name} top{n}", f"{set_name.capitalize()} Top-{n}"

        df = df.rename(columns={old_name: new_name})
        df[new_name] = df[new_name].apply(lambda x: f"{x * 100:.2f}")

# df

In [19]:
set_name = CUTOFF_9_NAMES[0]
subdf = df[df['subset'] == set_name]
print(f'{set_name}'.capitalize())

key = 'No. frames'
nfs = [16, 32]
for num_frames in [16, 32]:
    print(f'{key} : {num_frames}')
    df_nf = subdf[subdf[key] == num_frames]
    
    # display(subdf.sort_values('Test Top-1', ascending=False))
    # display(subdf.sort_values('best_val_loss', ascending=True))
    display(df_nf.sort_values('test_loss', ascending=True))

Asl100_cutoff_9
No. frames : 16


,model,exp no,run_id,No. frames,subset,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path,weight path
4,MViTv2_S,046,2rt70778,16,asl100_cutoff_9,63.18,88.37,95.35,70.41,89.64,95.27,71.893491,1.208193,1.464866,./configfiles/Satnac_2025/ViTs.toml,None
5,MViTv1_B,004,kk32z0vt,16,asl100_cutoff_9,58.91,85.27,92.25,60.06,86.98,92.01,65.384615,1.576679,1.722346,./configfiles/Satnac_2025/ViTs.toml,None
8,R3D_18,008,j61017cb,16,asl100_cutoff_9,53.10,80.62,90.31,58.88,85.50,90.53,61.834320,1.628094,1.909940,./configfiles/Satnac_2025/ResNet3D_f_16.toml,None
6,R(2+1)D_18,008,ef0nqw2i,16,asl100_cutoff_9,51.94,81.40,89.15,63.91,86.69,91.42,68.639053,1.457942,1.996388,./configfiles/Satnac_2025/ResNet3D_f_16.toml,None
1,Swin3D_B,004,6tpshfur,16,asl100_cutoff_9,56.98,85.66,93.02,63.02,86.09,93.20,67.159763,1.644104,2.003825,./configfiles/Satnac_2025/ViTs.toml,None
3,Swin3D_T,004,6m2t3qmj,16,asl100_cutoff_9,54.26,81.78,89.92,59.76,84.91,91.72,66.272189,1.723813,2.120142,./configfiles/Satnac_2025/ViTs.toml,None
2,Swin3D_S,009,70tm9wxo,16,asl100_cutoff_9,52.33,79.84,88.37,58.28,83.73,91.42,64.792899,1.858755,2.280269,./configfiles/Satnac_2025/ViTs.toml,None
10,S3D,089,jl96mldj,16,asl100_cutoff_9,34.88,70.16,81.40,47.34,72.49,82.84,52.071006,2.738303,3.258514,./configfiles/Satnac_2025/S3D_16.toml,None


No. frames : 32


,model,exp no,run_id,No. frames,subset,Test Top-1,Test Top-5,Test Top-10,Val Top-1,Val Top-5,Val Top-10,best_val_acc,best_val_loss,test_loss,config path,weight path
7,R3D_18,009,vjgqhpvm,32,asl100_cutoff_9,62.40,87.21,91.47,68.34,89.64,94.38,70.414201,1.344758,1.676170,./configfiles/Satnac_2025/ResNet3D_f_32.toml,None
0,R(2+1)D_18,010,wut7i25d,32,asl100_cutoff_9,53.49,79.07,88.76,63.61,85.80,91.12,65.976331,1.572321,2.393092,./configfiles/Satnac_2025/ResNet3D_f_32.toml,None
9,S3D,090,mg1rxecr,32,asl100_cutoff_9,41.47,73.26,83.72,48.22,77.22,87.87,55.621302,2.326178,2.854949,./configfiles/Satnac_2025/S3D_32.toml,None


In [ ]:


# Column used to select the best run per model/subset
from typing import Any

SELECTION_COL = "best_val_acc"   # change to "best_val_loss" if needed

# Optional: if you want custom model names with \cite, set escape=False below
# and store the LaTeX string directly in the 'model' column.
# ==========================================================

def make_latex_table(df : pd.DataFrame, key : str, value : Any, set_name: str):
    """Select best run per model and return LaTeX table for one subset."""
    sub = df[df[key] == value].copy()
    sub["model"] = sub["model"].astype(str).str.strip()

    # Convert selection column to numeric and drop rows with missing value
    sub[SELECTION_COL] = pd.to_numeric(sub[SELECTION_COL], errors="coerce")

    # Select best run: highest accuracy or lowest loss
    if SELECTION_COL == "best_val_loss":
        idx = sub.groupby("model")[SELECTION_COL].idxmin()
    else:
        idx = sub.groupby("model")[SELECTION_COL].idxmax()
    best = sub.loc[idx].copy()

    # Keep only needed columns and rename
    best = best[["model", "Test Top-1", "Test Top-5", "Test Top-10"]].rename( #type: ignore (dumbass pandas)
        columns={
            "model": "Model",
            "Test Top-1": "Acc@1",
            "Test Top-5": "Acc@5",
            "Test Top-10": "Acc@10",
        }
    )

    # Ensure metrics are numeric (to_latex will format them)
    for col in ["Acc@1", "Acc@5", "Acc@10"]:
        best[col] = pd.to_numeric(best[col], errors="coerce")

    # Generate LaTeX using .to_latex()
    latex = best.to_latex(
        index=False,
        na_rep="-",
        float_format="%.2f",
        caption=f"{set_name} Results",
        label=f"tab:{set_name.lower().replace('-', '_')}",
        position="htbp",
    )
    return latex


# Generate and print each table

    
print(f"% ===== Table for {set_name} =====")
for num_frames in nfs:
    print(make_latex_table(df, key,num_frames, set_name))
print("\n")   # blank line between tables

% ===== Table for asl100_cutoff_9 =====
\begin{table}[htbp]
\caption{asl100_cutoff_9 Results}
\label{tab:asl100_cutoff_9}
\begin{tabular}{lrrr}
\toprule
Model & Acc@1 & Acc@5 & Acc@10 \\
\midrule
MViTv1_B & 58.91 & 85.27 & 92.25 \\
MViTv2_S & 63.18 & 88.37 & 95.35 \\
R(2+1)D_18 & 51.94 & 81.40 & 89.15 \\
R3D_18 & 53.10 & 80.62 & 90.31 \\
S3D & 34.88 & 70.16 & 81.40 \\
Swin3D_B & 56.98 & 85.66 & 93.02 \\
Swin3D_S & 52.33 & 79.84 & 88.37 \\
Swin3D_T & 54.26 & 81.78 & 89.92 \\
\bottomrule
\end{tabular}
\end{table}

\begin{table}[htbp]
\caption{asl100_cutoff_9 Results}
\label{tab:asl100_cutoff_9}
\begin{tabular}{lrrr}
\toprule
Model & Acc@1 & Acc@5 & Acc@10 \\
\midrule
R(2+1)D_18 & 53.49 & 79.07 & 88.76 \\
R3D_18 & 62.40 & 87.21 & 91.47 \\
S3D & 41.47 & 73.26 & 83.72 \\
\bottomrule
\end{tabular}
\end{table}



